In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
df = pd.read_csv('OnlineRetail.csv', encoding='ISO-8859-1')

df.head()
print(df.shape)
df.head()

In [ ]:
df = df.dropna(subset=['CustomerID']).copy()
df_sales = df[df['Quantity'] != 0]
df_sales['CustomerID'] = df_sales['CustomerID'].astype(int).astype(str)
df_sales['InvoiceDate'] = pd.to_datetime(df_sales['InvoiceDate'])
df_sales['NetRevenue'] = df_sales['Quantity'] * df_sales['UnitPrice']
df_sales.head(10)



In [ ]:
df_sales = df_sales.groupby(
    ['InvoiceNo', 'StockCode', 'CustomerID', 'InvoiceDate'], as_index=False
).agg({
    'Quantity': 'sum',
    'NetRevenue': 'sum',
    'Country': 'first',
    'UnitPrice': 'mean'
    
})
df_sales.head(10)

In [ ]:
df_sales['InvoiceMonth'] = df_sales['InvoiceDate'].dt.to_period('M')
df_sales['CohortMonth'] = df_sales.groupby('CustomerID')['InvoiceMonth'].transform('min')
df_sales['CohortIndex'] = (df_sales['InvoiceMonth'] - df_sales['CohortMonth']).apply(lambda x: x.n)

In [ ]:
cohort_data = df_sales.groupby(['CohortMonth', 'CohortIndex'])['CustomerID'].nunique().reset_index()
cohort_data
cohort_counts = cohort_data.pivot(index='CohortMonth', columns='CohortIndex', values='CustomerID')
cohort_size = cohort_counts.iloc[:, 0]
retention = cohort_counts.divide(cohort_size, axis=0)
retention.round(2)

In [ ]:
plt.figure(figsize=(16, 8))
sns.heatmap(
            retention,
            annot=True,
            cmap='YlGnBu',
            fmt='.1%',
            vmin=0.0,
            vmax=0.5,
            linewidths=0.5
)
plt.title('Когортный анализ: Удержание клиентов (Retention Rate)', fontsize=16, fontweight='bold', pad=20)
plt.xlabel('Месяцы жизни когорты (Cohort Index)\n*Месяц 0 — первый месяц совершения заказа*', fontsize=12, labelpad=10)
plt.ylabel('Месяц привлечения (Cohort Month)', fontsize=12, labelpad=10)
plt.yticks(rotation = 0)
plt.savefig('retention_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
cohort_revenue = df_sales.groupby(['CohortMonth', 'CohortIndex'])['NetRevenue'].sum().reset_index()
revenue_matrix = cohort_revenue.pivot(index = 'CohortMonth', columns = 'CohortIndex', values = 'NetRevenue')
cumulative_revenue = revenue_matrix.cumsum(axis=1)
ltv_matrix = cumulative_revenue.divide(cohort_size, axis=0)
ltv_matrix.round(2)

In [ ]:
plt.figure(figsize=(16, 10))
sns.heatmap(
            ltv_matrix,
            annot=True,
            cmap='YlGnBu',
            fmt='.2f',
            linewidths=0.5
)
plt.title('Когортный анализ: Накопительный LTV на одного клиента (GBP)', fontsize=16, fontweight='bold', pad=20)
plt.xlabel('Месяцы с момента первого заказа (Cohort Index)', fontsize=12, labelpad=10)
plt.ylabel('Месяц привлечения (Cohort Month)', fontsize=12, labelpad=10)
plt.yticks(rotation=0)
plt.savefig('ltv_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()